Here, I do the following test

1. Load the Q-SENSE basis states (PT method)
2. Taper and factorize them
3. Check if the tapered and factorized states correspond correctly to the original states.

In [1]:
import sys
sys.path.append('../')

from utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state
)
from utils_m1_seniority import (
    project_out_seniority_symmetries
)
from utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn,
    factorize_state,
    evaluate_fully_classical_factors
)
from openfermion import (
    jordan_wigner,
    get_sparse_operator
)

import numpy as np
import pickle

In [2]:
# load Q-SENSE basis states

molecule = 'n2'
bond_length = 1.0
filename = f'../{molecule}_data/Uext_CSF_for_Praveen_Smik_{bond_length}.dump'

with open(filename, 'rb') as f:
    (
    list_list_refCSF,
    list_list_Uext_mp2_CSF,
    list_list_Uext_mp2_ampld,
    list_list_Uext_opt_ampld,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)

if molecule == 'h2o':
    Hsparse = get_sparse_operator(Hqub, obt.shape[0])

# load relevant data (CSF, UCSF, W information) in a linear list

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

UCSF_tz_states = []
CSF_tz_states  = []
W_amplitudes   = []

for i, ucsf_list in enumerate(list_list_Uext_mp2_CSF):
    for j, ucsf in enumerate(ucsf_list):
        UCSF_tz_states.append(ucsf)
        CSF_tz_states.append(list_list_refCSF[i][j])
        W_amplitudes.append(list_list_Uext_mp2_ampld[i])

# obtain information needed to taper and factorize

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list         = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list         = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list         = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) for i in range(Nstates)]

#
#
#    Verify that everything has been done correctly
#
#

# Test 1: can I recover the original statevectors from the tapered statevectors

statevectors_recov = [decompress_state(tapered_statevectors[i].toarray()[0], configs[i]) for i in range(Nstates)]
for i in range(Nstates):
    assert np.allclose(statevectors_recov[i], statevectors[i].toarray()[0])

# Test 2: can I recover the tapered statevectors from the original statevectors

tapered_statevectors_recov = [expand_tensor_product_for_incomplete_qubit_set(psi_t_FD) for psi_t_FD in factorized_tapered_statevectors]
for i in range(Nstates):
    assert np.allclose(tapered_statevectors[i].toarray()[0], tapered_statevectors_recov[i])

In [ ]:
# check manually that the qubit-partitions obtained from the factorization are correct, based on (state_type, SV, SW, SN)

for index in range(Nstates):
    print(f'''
    Index : {index}

        Type    : {state_type_list[index]}
        SW      : {SW_list[index]}
        SV      : {SV_list[index]}
        SN      : {SN_list[index]}
        FD_keys : {list(factorized_tapered_statevectors[index].keys())}
    ''')


    Index : 0

        Type    : 1
        SW      : (0, 1, 7, 8, 9)
        SV      : ()
        SN      : (2, 3, 4, 5, 6)
        FD_keys : [(0, 1, 7, 8, 9), (2,), (3,), (4,), (5,), (6,)]
    

    Index : 1

        Type    : 1
        SW      : (0, 1, 7, 8, 9)
        SV      : ()
        SN      : (2, 3, 4, 5, 6)
        FD_keys : [(0, 1, 7, 8, 9), (2,), (3,), (4,), (5,), (6,)]
    

    Index : 2

        Type    : 1
        SW      : (0, 1, 7, 8, 9)
        SV      : ()
        SN      : (2, 3, 4, 5, 6)
        FD_keys : [(0, 1, 7, 8, 9), (2,), (3,), (4,), (5,), (6,)]
    

    Index : 3

        Type    : 1
        SW      : (0, 1, 7, 8, 9)
        SV      : ()
        SN      : (2, 3, 4, 5, 6)
        FD_keys : [(0, 1, 7, 8, 9), (2,), (3,), (4,), (5,), (6,)]
    

    Index : 4

        Type    : 1
        SW      : (0, 1, 7, 8, 9)
        SV      : ()
        SN      : (2, 3, 4, 5, 6)
        FD_keys : [(0, 1, 7, 8, 9), (2,), (3,), (4,), (5,), (6,)]
    

    Index : 5

    

In [ ]:
# verify that the Hamiltonians you get using the three types of states agree -> this can only be done for H2O (N2 requires 21 qubits -> expensive)

Hsub_full = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    bra = statevectors[i]
    ket = statevectors[i]
    Hsub_full[i,i] = (bra @ Hsparse @ ket.T)[0,0]

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            bra = statevectors[i]
            ket = statevectors[j]
            Hsub_full[i,j] = (bra @ Hsparse @ ket.T)[0,0]
            Hsub_full[j,i] = (ket @ Hsparse @ bra.T)[0,0]
            assert np.allclose(Hsub_full[i,j], Hsub_full[j,i])

Hsub_tapered = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f"Tapered: {i, i}", end='\r')
    bra_t             = tapered_statevectors[i]
    ket_t             = tapered_statevectors[i]
    Htapered          = project_out_seniority_symmetries(Hqub, Nqubits, configs[i], configs[i])
    Htapered_sparse   = get_sparse_operator(Htapered, Norb)
    Hsub_tapered[i,i] = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]
    
for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f"Tapered: {i, j}", end='\r')
            bra_t             = tapered_statevectors[i]
            ket_t             = tapered_statevectors[j]
            Htapered          = project_out_seniority_symmetries(Hqub, Nqubits, configs[i], configs[j])
            Htapered_sparse   = get_sparse_operator(Htapered, Norb)
            Hsub_tapered[i,j] = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]
            Hsub_tapered[j,i] = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]
            
Hsub_factorized = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f"Factorized: {i, i}", end='\r')

    bra_f                       = factorized_tapered_statevectors[i]
    bra_labels                  = UCSF_information[i][0]
    ket_f                       = factorized_tapered_statevectors[i]
    ket_labels                  = UCSF_information[i][0]

    Htapered                    = project_out_seniority_symmetries(Hqub, Nqubits, configs[i], configs[i])

    HtQ, bratQ, kettQ, NqubitsQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)

    if NqubitsQ > 0:
        bratQ                       = convert_dense_format_to_sparse_format(bratQ)
        kettQ                       = convert_dense_format_to_sparse_format(kettQ)
        HtQ_sparse                  = get_sparse_operator(HtQ, NqubitsQ)
        Hsub_factorized[i,i]        = (bratQ @ HtQ_sparse @ kettQ.T)[0,0]

    else:
        Hsub_factorized[i,i]        = HtQ.constant

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f"Factorized: {i, j}", end='\r')

            bra_f                       = factorized_tapered_statevectors[i]
            bra_labels                  = UCSF_information[i][0]
            ket_f                       = factorized_tapered_statevectors[j]
            ket_labels                  = UCSF_information[j][0]

            Htapered                    = project_out_seniority_symmetries(Hqub, Nqubits, configs[i], configs[j])

            HtQ, bratQ, kettQ, NqubitsQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)

            if NqubitsQ == 0:
                Hsub_factorized[i,j] = HtQ.constant
                Hsub_factorized[j,i] = HtQ.constant

            else:
                bratQ                = convert_dense_format_to_sparse_format(bratQ)
                kettQ                = convert_dense_format_to_sparse_format(kettQ)
                HtQ_sparse           = get_sparse_operator(HtQ, NqubitsQ)
                Hsub_factorized[i,j] = (bratQ @ HtQ_sparse @ kettQ.T)[0,0]
                Hsub_factorized[j,i] = (bratQ @ HtQ_sparse @ kettQ.T)[0,0]


In [ ]:
# check if all three subspace Hamiltonians are the same

print(np.linalg.norm(Hsub_full - Hsub_tapered))
print(np.linalg.norm(Hsub_full - Hsub_factorized))
print(np.linalg.norm(Hsub_tapered - Hsub_factorized))

6.327180410021596e-13
6.32138053455569e-13
1.191884748736701e-13
